MNIST 

28x28 images of handwritten digits dataset. We want a vector as the example input. So, the inupt matrix would be [batch, image]

We use .view to flatten the image into a 784 sized vector. The input matrix is of size [batch, 784] (mental note about flattening: we don't need to worry about losing dimensionality or information as all examples are flattened in the same manner. Internally, python processes them as a flat array anways (I think)).

In [ ]:
import gzip
import torch
import struct
import numpy as np
import random 

def load_images(path):
    with gzip.open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows, cols)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data


images = load_images('gzip/emnist-digits-train-images-idx3-ubyte.gz')
labels = load_labels('gzip/emnist-digits-train-labels-idx1-ubyte.gz')

images = torch.tensor(images)
labels = torch.tensor(labels)

print(images.shape)
print(labels.shape)

print(images[:1, :, :])
print(labels[0])

torch.Size([240000, 28, 28])
torch.Size([240000])
tensor([[[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
            0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
            0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
            0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
            0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
            0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
            0,   0,   0,   0,   0,   0,   2,  21,  37,  21,   3, 

In [3]:
class Node():
    def __init__(self, value, parents=()):
        self.value = value
        self.grad = 0.0
        self.parents = set(parents)
        self.backward = lambda: None

    def __repr__(self):
        return ("Value(data={self.data})")
    
    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.value + other.value, (self, other))

        def backward():
            self.grad += output.grad
            other.grad += output.grad
        output.backward = backward
        return output 

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.value * other.value, (self, other))

        def backward():
            self.grad += output.grad * other.value
            other.grad += output.grad * self.value
        output.backward = backward
        return output
    
    def tanh(self):
        output = Node(np.tanh(self.value), (self,))

        def backward():
            self.grad += output.grad * (1.0 - output.value**2)
        output.backward = backward
        return output
    
    def call_backward(self):
        # Phase 1: Build topological order
        topological = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build_topo(parent)
                topological.append(node)

        build_topo(self)

        # Phase 2: Iterate in reverse, calling backward
        self.grad = 1.0
        for node in reversed(topological):
            node.backward()

In [9]:
#Manual Forward Pass 
x1 = Node(2.5)
w1 = Node(1.0)

x2 = Node(3.0) 
w2 = Node(-2.0)

a1 = x1 * w1
a2 = x2 * w2

b = a1 + a2

c = b.tanh()

#Call the Backward Pass
c.call_backward()

print(c.grad)
print(b.grad)
print(a1.grad)
print(a2.grad)
print(w1.grad)
print(w2.grad)
print(x1.grad)
print(x2.grad)


1.0
0.003640884720487403
0.003640884720487403
0.003640884720487403
0.009102211801218507
0.010922654161462209
0.003640884720487403
-0.007281769440974806


At this point we have the Node class setup, we have the manual forward pass and the autobackward pass. These operations operate on the individual scalar values so if we have vector and matrix operations we need to apply these elementwise to carry out the bigger operations. In the case of EMNIST the vectorization of the input image is a [28,28] vector which we flatten by representing in a 1D vector that's [784]. This would be 784 neurons in the input layer where each neuron represents the feeatures for all examples being passed to the hidden layer. We want to apply the weights to the features and not examples--we are training on features NOT examples. 

The weights are parameters that determine the relationship between the inputs and the expected outcomes. The architecture determines the family of relationships the model is capable of expressing. The weights determine how much the features contribute to the error where as if we had the examples as neurons we would determine how much a specific example determines the error but that doesn't generalize. Given some set of weights, they determine how much a feature should contribute to the final output, in a simple example of predicting house prices, we have features (sqft, beds, zip), given some training data the weights contributing to the beds might not contribute as much as the weight for the sqft for the price. 

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.weights = [Node(torch.randn(1).item() * (1/(nin)**0.5)) for _ in range(nin)] #Xavier/Glorot scaling 1/sqrt(nin) for smart initialization
        self.bias = Node(0)

    def activation(self, inputs):
        dotprod = sum((w * x for w, x in zip(self.weights, inputs)), self.bias)
        return dotprod.tanh()
    

class Layer:
    def __init__(self, nin, non):
        self.neurons = [Neuron(nin) for _ in range(non)]

    def forward(self, inputs):
        

class MLP:
    def __init__(self, nin, layers):
        layer_list = [nin] + layers
        for x, y in zip(layer_list, layer_list[1:]):
            Layer(x, y)

    def call_forward(self, input_matrix):




SyntaxError: incomplete input (31297242.py, line 15)